In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import itertools
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy import stats
import os, warnings

warnings.filterwarnings('ignore') # Turns off warnings

In [2]:
precip_data = pd.read_csv('combined_ENSO_highresprecip.csv')
df = pd.read_csv('DATABASE.csv')
sq = pd.read_csv('processed_swe_data.csv')


In [3]:
# Define state abbreviations and full names
state_dict = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY', 'District of Columbia': 'DC',
    'Puerto Rico': 'PRCP'
}

# Define the state dictionary
state_dictAB = {
    1: 'AL', 2: 'AK', 4: 'AZ', 5: 'AR', 6: 'CA', 8: 'CO', 9: 'CT', 10: 'DE',
    11: 'DC', 12: 'FL', 13: 'GA', 15: 'HI', 16: 'ID', 17: 'IL', 18: 'IN',
    19: 'IA', 20: 'KS', 21: 'KY', 22: 'LA', 23: 'ME', 24: 'MD', 25: 'MA',
    26: 'MI', 27: 'MN', 28: 'MS', 29: 'MO', 30: 'MT', 31: 'NE', 32: 'NV',
    33: 'NH', 34: 'NJ', 35: 'NM', 36: 'NY', 37: 'NC', 38: 'ND', 39: 'OH',
    40: 'OK', 41: 'OR', 42: 'PA', 44: 'RI', 45: 'SC', 46: 'SD', 47: 'TN',
    48: 'TX', 49: 'UT', 50: 'VT', 51: 'VA', 53: 'WA', 54: 'WV', 55: 'WI',
    56: 'WY'
}

# Iterate through states
for state, state_abbr in state_dictAB.items():
    precip_column = f'Precip_Anomaly_{state_abbr}'

    if precip_column not in precip_data.columns:
        print(f"No precipitation anomaly data exists for state: {state_abbr}, passing...")
        continue

    # Standardize the precipitation anomaly data
    precip_data[precip_column] = (precip_data[precip_column] - precip_data[precip_column].mean()) / precip_data[precip_column].std()



No precipitation anomaly data exists for state: AK, passing...
No precipitation anomaly data exists for state: DC, passing...
No precipitation anomaly data exists for state: HI, passing...


In [4]:


# Iterate through states
for state, state_abbr in state_dictAB.items():
    precip_column = f'Precip_Anomaly_{state_abbr}'

    if precip_column not in precip_data.columns:
        print(f"No precipitation anomaly data exists for state: {state_abbr}, passing...")
        continue

    # Calculate ENSO-related precipitation anomalies
    x_values = precip_data['ENSO_Value']
    y_values = precip_data[precip_column]
    
    # Linear regression to get slope and intercept for ENSO-related precipitation anomalies
    slope, intercept, r_value, p_value, std_err = stats.linregress(x_values, y_values)
    precip_data[f'P_enso_{state_abbr}'] = slope * precip_data['ENSO_Value'] + intercept

   # Compute non-ENSO-related precipitation anomalies
    precip_data[f'P_nonenso_{state_abbr}'] = precip_data[precip_column] - precip_data[f'P_enso_{state_abbr}']
    
    # precip_data[f'P_nonenso_{state_abbr}'] = precip_data[f'Total_Precip_Anomaly_{state_abbr}'] - precip_data[f'P_enso_{state_abbr}']

    


No precipitation anomaly data exists for state: AK, passing...
No precipitation anomaly data exists for state: DC, passing...
No precipitation anomaly data exists for state: HI, passing...


In [5]:
# Initialize DataFrame to collect results
results_df = pd.DataFrame(columns=['State', 'Slope', 'Intercept', 'R_Value', 'P_Value', 'Std_Err', 'R_Squared'])

for state, state_abbr in state_dictAB.items():
    precip_column = f'Precip_Anomaly_{state_abbr}'

    if precip_column not in precip_data.columns:
        print(f"No precipitation anomaly data exists for state: {state_abbr}, skipping...")
        continue

    # Ensure the ENSO and non-ENSO columns exist
    if not (f'P_enso_{state_abbr}' in precip_data.columns and f'P_nonenso_{state_abbr}' in precip_data.columns):
        print(f"ENSO or non-ENSO columns missing for state: {state_abbr}, skipping...")
        continue

    # Create a new DataFrame to calculate total precipitation anomaly without modifying precip_data
    total_precip_data = precip_data.copy()
    total_precip_data[f'Total_Precip_Anomaly_{state_abbr}'] = total_precip_data[f'P_enso_{state_abbr}'] + total_precip_data[f'P_nonenso_{state_abbr}']

    # Filter crash data for the current state
    state_df = df[df['STATE'] == state]

    if state_df.empty:
        print(f"No crash data for state: {state_abbr}, skipping...")
        continue

    # Group and calculate crash counts and anomalies
    crash_counts = state_df.groupby(['YEAR', 'MONTH']).size().reset_index(name='CRASH_COUNT')
    monthly_avg_crashes = crash_counts.groupby('MONTH')['CRASH_COUNT'].mean()
    crash_counts['FWRCA'] = crash_counts.apply(lambda row: row['CRASH_COUNT'] - monthly_avg_crashes[row['MONTH']], axis=1)

    # Merge crash data with precipitation data
    temp_merged_df = crash_counts.merge(total_precip_data[['YEAR', 'Month', precip_column]], 
                                   left_on=['YEAR', 'MONTH'], 
                                   right_on=['YEAR', 'Month'], 
                                   how='inner')

    # Add total precipitation anomaly to temp_merged_df
    if f'Total_Precip_Anomaly_{state_abbr}' in total_precip_data.columns:
        temp_merged_df = temp_merged_df.merge(total_precip_data[['YEAR', 'Month', f'Total_Precip_Anomaly_{state_abbr}']], 
                                    on=['YEAR', 'Month'], 
                                    how='left')
    else:
        print(f"Total Precipitation Anomaly column missing for state: {state_abbr}, skipping...")
        continue

    # Ensure the merged DataFrame is not empty before proceeding
    if temp_merged_df.empty:
        print(f"No valid data for state: {state_abbr}, skipping...")
        continue

    # Perform linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(temp_merged_df[f'Total_Precip_Anomaly_{state_abbr}'], temp_merged_df['FWRCA'])
    new_row = pd.DataFrame([{
        'State': state_abbr,
        'Slope': slope,
        'Intercept': intercept,
        'R_Value': r_value,
        'P_Value': p_value,
        'Std_Err': std_err,
        'R_Squared': r_value**2
    }])
    
    results_df = pd.concat([results_df, new_row], ignore_index=True)

    # Plot
    plt.figure(figsize=(10, 6))
    plt.scatter(temp_merged_df[f'Total_Precip_Anomaly_{state_abbr}'], temp_merged_df['FWRCA'], color='blue', label=f'FWRCA vs Total Precip Anomaly {state_abbr}')
    plt.plot(temp_merged_df[f'Total_Precip_Anomaly_{state_abbr}'], 
             slope * temp_merged_df[f'Total_Precip_Anomaly_{state_abbr}'] + intercept, 
             color='red', label=f'Linear Regression (R²={r_value**2:.3f})')

    plt.xlabel(f'Total Precipitation Anomalies - {state_abbr}')
    plt.ylabel('Fatal Weather-Related Crash Anomalies (FWRCA)')
    plt.title(f'Total Precipitation Anomalies vs FWRCA in {state_abbr}')
    plt.legend()
    plt.grid(True)
    plt.savefig(f'Total_Precip_Anomaly_vs_FWRCA(Winter)_{state_abbr}_hires.png', facecolor='white', bbox_inches='tight')
    plt.show()


ValueError: Unable to parse string "JAN" at position 0

In [ ]:
# Loop through each state and calculate ENSO-related precipitation anomalies and plot
for state, state_abbr in state_dict.items():
    precip_column = f'Precip_Anomaly_{state_abbr}'
    
    if precip_column not in precip_data.columns:
        print(f"No precipitation anomaly data exists for state: {state_abbr}, passing...")
        continue
    
    if f'P_enso_{state_abbr}' not in precip_data.columns:
        print(f"No ENSO-related precipitation anomaly data for state: {state_abbr}, skipping...")
        continue
    
    # Filter the data for the current state and prepare for regression
    merged_df = precip_data[['YEAR', 'Month', 'ENSO_Value', f'P_enso_{state_abbr}']].copy()

    precip_data[f'P_enso_{state_abbr}'] = slope * precip_data['ENSO_Value'] + intercept
    # Perform linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(merged_df['ENSO_Value'], merged_df[f'P_enso_{state_abbr}'])



In [ ]:
# Loop through each state and calculate Total precipitation anomalies and plot for winter months
for state, state_abbr in state_dict.items():
    precip_column = f'Precip_Anomaly_{state_abbr}'
    
    if precip_column not in precip_data.columns:
        print(f"No precipitation anomaly data exists for state: {state_abbr}, passing...")
        continue
    
    if f'P_enso_{state_abbr}' not in precip_data.columns:
        print(f"No ENSO-related precipitation anomaly data for state: {state_abbr}, skipping...")
        continue
    
    # Filter the data for the current state and prepare for regression, keeping only winter months (December, January, February)
    merged_df = precip_data[['YEAR', 'Month', 'ENSO_Value', f'Total_Precip_Anomaly_{state_abbr}']].copy()

    # Filter for winter months (December: 12, January: 1, February: 2)
    merged_df = merged_df[merged_df['Month'].isin([12, 1, 2])]

    # Remove rows with NaN values in relevant columns
    merged_df = merged_df.dropna(subset=['ENSO_Value', f'Total_Precip_Anomaly_{state_abbr}'])

    # Ensure valid data for regression
    if merged_df.empty:
        print(f"No valid data for regression in state: {state_abbr}")
        continue

    # Perform linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(merged_df['ENSO_Value'], merged_df[f'Total_Precip_Anomaly_{state_abbr}'])

    # Plotting
    plt.figure(figsize=(10, 6))
    plt.scatter(merged_df['ENSO_Value'], merged_df[f'Total_Precip_Anomaly_{state_abbr}'], color='blue', label=f'Total Precip Anomaly vs SST {state_abbr}')
    
    # Plot the regression line
    line = slope * merged_df['ENSO_Value'] + intercept
    plt.plot(merged_df['ENSO_Value'], line, color='red', label=f'Linear Regression (R²={r_value**2:.3f})')

    # Plot details
    plt.xlabel('SST Anomalies (ENSO Value)')
    plt.ylabel(f'Total Precipitation Anomalies - {state_abbr}')
    plt.title(f'SST Anomalies vs Total Precipitation Anomalies in {state_abbr} (Winter Months)')
    plt.legend()
    plt.grid(True)

    # Save the plot
    plt.savefig(f'SST_Anomalies_vs_Total_Precip_Anomalies_Winter_{state_abbr}_highres.png', facecolor='white', bbox_inches='tight')
    plt.show()

In [ ]:
# Initialize lists to store the R² values and corresponding states
states = []
r_squared_values = []

# Loop through each state and calculate Total precipitation anomalies and plot for winter months
for state, state_abbr in state_dict.items():
    precip_column = f'Precip_Anomaly_{state_abbr}'
    
    if precip_column not in precip_data.columns:
        print(f"No precipitation anomaly data exists for state: {state_abbr}, passing...")
        continue
    
    if f'P_enso_{state_abbr}' not in precip_data.columns:
        print(f"No ENSO-related precipitation anomaly data for state: {state_abbr}, skipping...")
        continue
    
    # Filter the data for the current state and prepare for regression, keeping only winter months (December, January, February)
    merged_df = precip_data[['YEAR', 'Month', 'ENSO_Value', f'Total_Precip_Anomaly_{state_abbr}']].copy()

    # Filter for winter months (December: 12, January: 1, February: 2)
    merged_df = merged_df[merged_df['Month'].isin([12, 1, 2])]

    # Remove rows with NaN values in relevant columns
    merged_df = merged_df.dropna(subset=['ENSO_Value', f'Total_Precip_Anomaly_{state_abbr}'])

    # Ensure valid data for regression
    if merged_df.empty:
        print(f"No valid data for regression in state: {state_abbr}")
        continue

    # Perform linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(merged_df['ENSO_Value'], merged_df[f'Total_Precip_Anomaly_{state_abbr}'])
    
    # Append the state and R² value to the lists
    states.append(state_abbr)
    r_squared_values.append(r_value ** 2)

# Create a DataFrame from the collected states and R² values
import pandas as pd

r_squared_df = pd.DataFrame({'State': states, 'R_Squared': r_squared_values})

# Ensure the DataFrame is not empty
if not r_squared_df.empty:
    # Sort the DataFrame by R² in descending order
    sorted_r_squared_df = r_squared_df.sort_values(by='R_Squared', ascending=False)

    # Plotting a horizontal bar plot for R² values for each state
    plt.figure(figsize=(10, 12))
    plt.barh(sorted_r_squared_df['State'], sorted_r_squared_df['R_Squared'], color='lightpink', edgecolor='black')

    plt.xlabel('R² Value')
    plt.ylabel('State')
    plt.title('R² Values of Total Precipitation Anomalies vs SST Anomalies for Each State (Sorted by Correlation)')
    plt.grid(axis='x', linestyle='--', alpha=0.7)

    # Invert y-axis to have the largest R² at the top
    plt.gca().invert_yaxis()

    # Add a red dashed line at R² = 0.20
    plt.axvline(x=0.10, color='red', linestyle='--', linewidth=1, label='R² = 0.10 (Significance Threshold)')

    # Add legend
    plt.legend()

    # Save and show the plot
    plt.savefig('PRCPSST_hires.png', facecolor='white', bbox_inches='tight')
    plt.show()
else:
    print("No valid data available to plot the R² values.")

In [ ]:
# Initialize DataFrame to collect results
results_df = pd.DataFrame(columns=['State', 'Slope', 'Intercept', 'R_Value', 'P_Value', 'Std_Err', 'R_Squared'])

for state, state_abbr in state_dictAB.items():
    precip_column = f'Precip_Anomaly_{state_abbr}'

    if precip_column not in precip_data.columns:
        print(f"No precipitation anomaly data exists for state: {state_abbr}, skipping...")
        continue

    # Ensure the ENSO and non-ENSO columns exist
    if not (f'P_enso_{state_abbr}' in precip_data.columns and f'P_nonenso_{state_abbr}' in precip_data.columns):
        print(f"ENSO or non-ENSO columns missing for state: {state_abbr}, skipping...")
        continue

    # Create a new DataFrame to calculate total precipitation anomaly without modifying precip_data
    total_precip_data = precip_data.copy()
    total_precip_data[f'Total_Precip_Anomaly_{state_abbr}'] = total_precip_data[f'P_enso_{state_abbr}'] + total_precip_data[f'P_nonenso_{state_abbr}']

    # Filter crash data for the current state
    state_df = df[df['STATE'] == state]

    if state_df.empty:
        print(f"No crash data for state: {state_abbr}, skipping...")
        continue

    # Group and calculate crash counts and anomalies
    crash_counts = state_df.groupby(['YEAR', 'MONTH']).size().reset_index(name='CRASH_COUNT')
    monthly_avg_crashes = crash_counts.groupby('MONTH')['CRASH_COUNT'].mean()
    crash_counts['FWRCA'] = crash_counts.apply(lambda row: row['CRASH_COUNT'] - monthly_avg_crashes[row['MONTH']], axis=1)

    # Merge crash data with precipitation data
    temp_merged_df = crash_counts.merge(total_precip_data[['YEAR', 'Month', precip_column]], 
                                   left_on=['YEAR', 'MONTH'], 
                                   right_on=['YEAR', 'Month'], 
                                   how='inner')

    # Add total precipitation anomaly to temp_merged_df
    if f'Total_Precip_Anomaly_{state_abbr}' in total_precip_data.columns:
        temp_merged_df = temp_merged_df.merge(total_precip_data[['YEAR', 'Month', f'Total_Precip_Anomaly_{state_abbr}']], 
                                    on=['YEAR', 'Month'], 
                                    how='left')
    else:
        print(f"Total Precipitation Anomaly column missing for state: {state_abbr}, skipping...")
        continue

    # Ensure the merged DataFrame is not empty before proceeding
    if temp_merged_df.empty:
        print(f"No valid data for state: {state_abbr}, skipping...")
        continue

    # Perform linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(temp_merged_df[f'Total_Precip_Anomaly_{state_abbr}'], temp_merged_df['FWRCA'])
    new_row = pd.DataFrame([{
        'State': state_abbr,
        'Slope': slope,
        'Intercept': intercept,
        'R_Value': r_value,
        'P_Value': p_value,
        'Std_Err': std_err,
        'R_Squared': r_value**2
    }])
    
    results_df = pd.concat([results_df, new_row], ignore_index=True)

    # Plot
    plt.figure(figsize=(10, 6))
    plt.scatter(temp_merged_df[f'Total_Precip_Anomaly_{state_abbr}'], temp_merged_df['FWRCA'], color='blue', label=f'FWRCA vs Total Precip Anomaly {state_abbr}')
    plt.plot(temp_merged_df[f'Total_Precip_Anomaly_{state_abbr}'], 
             slope * temp_merged_df[f'Total_Precip_Anomaly_{state_abbr}'] + intercept, 
             color='red', label=f'Linear Regression (R²={r_value**2:.3f})')

    plt.xlabel(f'Total Precipitation Anomalies - {state_abbr}')
    plt.ylabel('Fatal Weather-Related Crash Anomalies (FWRCA)')
    plt.title(f'Total Precipitation Anomalies vs FWRCA in {state_abbr}')
    plt.legend()
    plt.grid(True)
    plt.savefig(f'Total_Precip_Anomaly_vs_FWRCA(Winter)_{state_abbr}_hires.png', facecolor='white', bbox_inches='tight')
    plt.show()

In [ ]:
# Ensure results_df is not empty
if not results_df.empty:
    # Sort the DataFrame by R² in descending order
    sorted_results_df = results_df.sort_values(by='R_Squared', ascending=False)

    # Plotting a horizontal bar plot for R² values for each state
    plt.figure(figsize=(10, 12))
    plt.barh(sorted_results_df['State'], sorted_results_df['R_Squared'], color='skyblue', edgecolor='black')

    plt.xlabel('R² Value')
    plt.ylabel('State')
    plt.title('R² Values of Total Precipitation Anomalies vs FWRCA for Each State (Sorted by Correlation)')
    plt.grid(axis='x', linestyle='--', alpha=0.7)

    # Invert y-axis to have the largest R² at the top
    plt.gca().invert_yaxis()

    # Add a red dashed line at R² = 0.20
    plt.axvline(x=0.20, color='red', linestyle='--', linewidth=1, label='R² = 0.20 (Significance Threshold)')

    # Add legend
    plt.legend()

    # Save and show the plot
    plt.savefig('PRCPFWRCA_hires', facecolor='white', bbox_inches='tight')
    plt.show()
else:
    print("No valid data available to plot the R² values.")